In [ ]:
# 주택 가격예측

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping
import matplotlib.pyplot as plt

# 1. 데이터 로드
file_path = '국토교통부_주택 공시가격 정보(2024).csv'
df = pd.read_csv(file_path)

# 필요한 열 선택
features = ['시도', '시군구', '전용면적']
target = '공시가격'

# 결측치 제거
df.dropna(subset=features + [target], inplace=True)

# 2. 원-핫 인코딩 (ColumnTransformer 대신)
X = df[features]
y = df[target]

# pandas의 get_dummies 활용 → 범주형 변수 자동 원-핫 인코딩
X = pd.get_dummies(X, columns=['시도', '시군구'], drop_first=True)

# 3. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#  추가 할 때 
# 3-1. 데이터 스케일링 추가
from sklearn.preprocessing import StandardScaler

# 입력 특성(X) 스케일링: '전용면적' 열만 찾아서 스케일링
# 원-핫 인코딩 후 '전용면적'은 보통 첫 번째 열로 남아있음
# 더 안전하게는 열 이름을 직접 지정하거나 Index를 활용해야 합니다.
# 여기서는 '전용면적' 열이 0번째 인덱스에 있다고 가정하고 진행합니다.


# 4-2. 출력 타겟(y) 스케일링: 공시가격 (MAE를 해석하기 쉽게 만들기 위해)
# MSE를 사용하기 때문에 타겟도 스케일링하면 학습이 안정적임
scaler_y = StandardScaler()
# y_train과 y_test는 Series이므로 numpy 배열로 변환 후 2차원 형태로 reshape 필요
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)) # 전치 행렬이 됨
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))


# 4. 텐서플로우 모델 구축
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

# 모델 컴파일
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 모델 요약
model.summary()

# 5. 모델 학습
modelpath="house_price_model.h5"
checkpointer = ModelCheckpoint(filepath=modelpath, monitor='val_loss', verbose=1, save_best_only=True)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(
    X_train,  y_train_scaled , # y_train
    epochs=100, # 충분히 큰 값으로 설정 (Early Stopping이 자동으로 종료시키도록)
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    # 콜백 리스트 추가
    callbacks=[early_stopping_callback, checkpointer] 
)


# 학습 완료 후 모델 저장
#model.save("house_price_model.h5")
#print("모델이 저장되었습니다.")

In [ ]:
#  재 학습
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping
import matplotlib.pyplot as plt

# 1. 데이터 로드
file_path = '국토교통부_주택 공시가격 정보(2024).csv'
df = pd.read_csv(file_path)

# 필요한 열 선택
features = ['시도', '시군구', '전용면적']
target = '공시가격'

# 결측치 제거
df.dropna(subset=features + [target], inplace=True)

# 2. 원-핫 인코딩 (ColumnTransformer 대신)
X = df[features]
y = df[target]

# pandas의 get_dummies 활용 → 범주형 변수 자동 원-핫 인코딩
X = pd.get_dummies(X, columns=['시도', '시군구'], drop_first=True)

# 3. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#  추가 할 때 
# 3-1. 데이터 스케일링 추가
from sklearn.preprocessing import StandardScaler

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# 4. 모델 불러오기
model = tf.keras.models.load_model("house_price_model.h5")
print("저장된 모델을 불러왔습니다.")

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae']) 

# 5. 모델 학습
modelpath="house_price_model.h5"
checkpointer = ModelCheckpoint(filepath=modelpath, monitor='val_loss', verbose=1, save_best_only=True)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(
    X_train, y_train_scaled , # y_train,
    epochs=100, # 충분히 큰 값으로 설정 (Early Stopping이 자동으로 종료시키도록)
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    # 콜백 리스트 추가
    callbacks=[early_stopping_callback, checkpointer] 
)



# 학습 완료 후 모델 저장
#model.save("house_price_model.h5")
#print("모델이 저장되었습니다.")

In [ ]:
# 예측 코드
# 8. 예측 예시
# 예측할 데이터 정의
data_new = {
    '시도': ['서울특별시'],
    '시군구': ['종로구'],
    '전용면적': [187.4]
}
df_new = pd.DataFrame(data_new)

# 학습 데이터(X)의 컬럼 목록을 가져옴
train_cols = X.columns.tolist() 

# 1. 원-핫 인코딩 적용 (get_dummies)
X_new = pd.get_dummies(df_new, columns=['시도', '시군구'], drop_first=True)

# 2. 학습 데이터의 컬럼 구조 맞추기
# 학습 데이터에만 존재하는 컬럼은 0으로 채우고, 예측 데이터에만 있는 컬럼은 제거
X_new_aligned = X_new.reindex(columns=train_cols, fill_value=0)

# '전용면적' 스케일링 (학습 시 사용한 scaler_X 사용)
# '전용면적' 컬럼만 스케일링
feature_to_scale = '전용면적'

if feature_to_scale in X_new_aligned.columns:
    # X_new_aligned의 '전용면적'에 대해 transform 적용
    X_new_scaled_part = scaler_X.transform(X_new_aligned[[feature_to_scale]])
    
    # 스케일링된 데이터를 X_new_aligned에 반영
    X_new_aligned[feature_to_scale] = X_new_scaled_part.flatten()
    print(f"예측 데이터 '{feature_to_scale}' 스케일링 완료.")
else:
    print(f"예측 데이터에 '{feature_to_scale}' 컬럼이 없습니다. 스케일링을 건너뜁니다.")


# 3. 모델 로드 (가장 잘 학습된 모델 사용)
from tensorflow.keras.models import load_model

# 학습 시 저장한 모델 경로 사용
best_model = load_model(modelpath) 

# 4. 예측 수행 (스케일링된 입력 사용)
y_pred_scaled = best_model.predict(X_new_aligned)

# 5. 스케일링된 예측값을 원래 공시가격 단위로 복원 (scaler_y 사용)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 결과 출력
print("-" * 50)
print(f"입력 조건: 시도='서울특별시', 시군구='종로구', 전용면적={data_new['전용면적'][0]}")
print(f"예측된 공시가격: {y_pred[0][0]:,.0f} (원)")
print("-" * 50)